# Pose tracking – SoccerNet tracking-2023 (SNMOT-137)
Ejecutar las celdas en orden (o *Entorno de ejecución → Ejecutar todas*). Usar un entorno con GPU.

El trabajo pesado (ZIP de 8.7 GB, vídeo de trabajo, resultados) se hace en el disco local de Colab.
En Drive solo se guardan archivos pequeños: el vídeo del clip, sus anotaciones y los resultados finales.

In [7]:
from google.colab import drive
import os

drive.mount('/content/drive')

ruta_repo = '/content/drive/MyDrive/football/football'
os.chdir(ruta_repo)
print(f"Directorio de trabajo actual: {os.getcwd()}")

!pip install -r requirements.txt -q
!pip install SoccerNet --upgrade -q
!pip install "lap>=0.5.12" -q   # lo necesita el tracker de Ultralytics; instalarlo aquí evita el aviso de "restart runtime"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Directorio de trabajo actual: /content/drive/MyDrive/football/football


In [8]:
# --- Configuración: todas las rutas y parámetros en un solo sitio ---
import shutil

SEQ = "SNMOT-137"
SCRIPT_PATH = "scripts/pose_tracking_baseline.py"

# Disco local de Colab: rápido y fiable para el trabajo pesado (se borra al cerrar la sesión).
# Leer el vídeo directamente desde Drive provocaba "Stream timeout ... Failed to open video".
local_dir = "/content/soccernet"
zip_path = os.path.join(local_dir, "tracking-2023", "test.zip")   # 8.7 GB, fuera de tu cuota de Drive
video_local = f"/content/{SEQ}.mp4"
output_local = f"/content/outputs/{SEQ}"

# Drive: solo archivos pequeños que quieres conservar entre sesiones
data_dir = os.path.join(ruta_repo, "data", "raw")
video_drive = os.path.join(data_dir, f"{SEQ}.mp4")
gt_drive = os.path.join(data_dir, f"{SEQ}_gt")
outputs_drive = os.path.join(ruta_repo, "outputs", SEQ)

os.makedirs(local_dir, exist_ok=True)
os.makedirs(data_dir, exist_ok=True)

libre_gb = shutil.disk_usage("/content/drive/MyDrive").free / 1e9
print(f"Espacio libre en Drive: {libre_gb:.1f} GB")
if libre_gb < 1:
    print("⚠️ Drive casi lleno: las escrituras en Drive pueden fallar o colgarse. "
          "Borra lo que no necesites (p. ej. data/raw/tracking-2023) y vacía la papelera de Drive.")

Espacio libre en Drive: 7.9 GB


In [9]:
import zipfile
import subprocess
from pathlib import Path

import cv2
from tqdm import tqdm


def video_ok(path) -> bool:
    """El vídeo existe y OpenCV puede abrirlo y leer un frame."""
    if not os.path.exists(path):
        return False
    cap = cv2.VideoCapture(str(path))
    ok = cap.isOpened() and cap.get(cv2.CAP_PROP_FRAME_COUNT) > 0 and cap.read()[0]
    cap.release()
    return bool(ok)


def construir_video_desde_zip() -> None:
    # 1. Descargar el ZIP de SoccerNet en disco local (no en Drive)
    if not os.path.exists(zip_path):
        from SoccerNet.Downloader import SoccerNetDownloader
        downloader = SoccerNetDownloader(LocalDirectory=local_dir)
        downloader.password = "s0cc3rn3t"
        downloader.downloadDataTask(task="tracking-2023", split=["test"])
    if not os.path.exists(zip_path):
        raise FileNotFoundError(f"La descarga no dejó el ZIP en {zip_path}")

    # 2. Extraer SOLO la secuencia que necesitamos, no todo el split
    extract_dir = Path("/content/temp_tracking")
    with zipfile.ZipFile(zip_path) as zf:
        miembros = [m for m in zf.namelist() if f"{SEQ}/" in m]
        if not miembros:
            raise FileNotFoundError(f"{SEQ} no está dentro de {zip_path}")
        for m in tqdm(miembros, desc=f"Extrayendo {SEQ}", unit="archivo"):
            zf.extract(m, extract_dir)
    seq_dir = next(extract_dir.rglob(f"{SEQ}/img1")).parent

    # 3. Guardar en Drive las anotaciones (pocos KB) para evaluar el tracking más adelante
    os.makedirs(gt_drive, exist_ok=True)
    for rel in ["gt/gt.txt", "gameinfo.ini", "seqinfo.ini"]:
        if (seq_dir / rel).exists():
            shutil.copy(seq_dir / rel, gt_drive)

    # 4. Imágenes -> vídeo, en disco local (+faststart: metadatos al principio del archivo)
    subprocess.run([
        "ffmpeg", "-y", "-loglevel", "error",
        "-framerate", "25", "-start_number", "1",
        "-i", str(seq_dir / "img1" / "%06d.jpg"),
        "-c:v", "libx264", "-crf", "18", "-pix_fmt", "yuv420p", "-movflags", "+faststart",
        video_local,
    ], check=True)
    shutil.rmtree(extract_dir, ignore_errors=True)

    # 5. Copia de seguridad en Drive, para no repetir la descarga en otra sesión
    try:
        shutil.copy(video_local, video_drive)
        print(f"Copia de seguridad en Drive: {video_drive}")
    except OSError as e:
        print(f"⚠️ No se pudo copiar el vídeo a Drive ({e}). El vídeo local sirve para esta sesión.")


if video_ok(video_local):
    print(f"El vídeo ya está en disco local: {video_local}")
else:
    if os.path.exists(video_drive):
        try:
            shutil.copy(video_drive, video_local)   # copia secuencial: más fiable que leer desde Drive con OpenCV
        except OSError as e:
            print(f"⚠️ No se pudo copiar desde Drive ({e}).")
    if video_ok(video_local):
        print(f"Vídeo copiado de Drive a {video_local}")
    else:
        construir_video_desde_zip()
        if not video_ok(video_local):
            raise RuntimeError(f"El vídeo generado no se puede abrir: {video_local}")

print(f"Vídeo listo: {video_local}")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Extrayendo SNMOT-137: 100%|██████████| 758/758 [00:06<00:00, 109.19archivo/s]


Copia de seguridad en Drive: /content/drive/MyDrive/football/football/data/raw/SNMOT-137.mp4
Vídeo listo: /content/SNMOT-137.mp4


In [10]:
import re
from pathlib import Path

script_file = Path(ruta_repo) / SCRIPT_PATH
code = script_file.read_text(encoding="utf-8-sig")   # -sig: ignora un BOM si lo hay

def quitar_markdown(texto):
    """Devuelve solo el código del primer bloque ```...```, aunque falte el cierre
    o haya texto antes/después del bloque."""
    lineas = texto.splitlines()
    es_fence = lambda l: l.strip().startswith("```")
    inicio = next((i for i, l in enumerate(lineas) if es_fence(l)), None)
    if inicio is None:
        return texto
    fin = next((i for i in range(inicio + 1, len(lineas)) if es_fence(lineas[i])), len(lineas))
    cuerpo = "\n".join(lineas[inicio + 1:fin])
    cuerpo = re.sub(r"\s*```\s*$", "", cuerpo)       # cierre pegado a la última línea de código
    return cuerpo.rstrip() + "\n"

# Si el script se guardó con etiquetas ```python (p. ej. copiado de un chat), quitarlas.
# Solo se toca el archivo si no compila tal cual.
try:
    compile(code, str(script_file), "exec")
    print("Sintaxis del script OK, no hace falta limpiarlo.")
except SyntaxError:
    limpio = quitar_markdown(code)
    try:
        compile(limpio, str(script_file), "exec")
    except SyntaxError:
        print("Tras quitar las etiquetas sigue sin compilar. Últimas líneas del script:")
        print("\n".join(limpio.splitlines()[-5:]))
        raise
    script_file.write_text(limpio, encoding="utf-8")
    print(f"Etiquetas Markdown eliminadas de {SCRIPT_PATH}. Sintaxis OK.")

Sintaxis del script OK, no hace falta limpiarlo.


In [11]:
# Entrada y salida en disco local; la celda siguiente copia los resultados a Drive.
# Notas sobre los parámetros:
# - Los frames son 1920x1080. imgsz 2560 reescala x1.33 (~1.8x más cómputo y memoria que 1920).
#   Si hay OOM o va muy lento, prueba --imgsz 1920.
# - Con el script revisado puedes añadir --half (FP16, ~2x más rápido en GPU) y probar --model yolo26s-pose.pt.
!python "{SCRIPT_PATH}" \
    --source "{video_local}" \
    --output-dir "{output_local}" \
    --imgsz 2560 \
    --conf 0.05 \
    --tracker botsort.yaml

WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
Procesando vídeo: 100% 750/750 [02:42<00:00,  4.61frame/s]

Processed 750 frames (1920x1080 @ 25.0fps, imgsz=2560).
Frames with >=1 player detected: 741/750 (98.8%)
Tracked players per frame (mean): 10.9
Unique track IDs seen: 57
Median track length: 87 frames; tracks shorter than 1 s: 13 (many short tracks = fragmentation / ID switches)
Annotated video -> /content/outputs/SNMOT-137/SNMOT-137_tracked.mp4
Keypoints JSON  -> /content/outputs/SNMOT-137/SNMOT-137_keypoints.json
MOT tracks      -> /content/outputs/SNMOT-137/SNMOT-137_mot.txt


In [12]:
# Copiar los resultados a Drive
from pathlib import Path

resultados = sorted(Path(output_local).glob("*")) if os.path.isdir(output_local) else []
if not resultados:
    print(f"No hay resultados en {output_local}; revisa la salida de la celda anterior.")
os.makedirs(outputs_drive, exist_ok=True)
for f in resultados:
    try:
        shutil.copy(f, outputs_drive)
        print(f"{f.name} -> {outputs_drive}")
    except OSError as e:
        print(f"⚠️ {f.name} no se pudo copiar a Drive ({e}). Descárgalo desde el panel Archivos: {f}")

SNMOT-137_keypoints.json -> /content/drive/MyDrive/football/football/outputs/SNMOT-137
SNMOT-137_mot.txt -> /content/drive/MyDrive/football/football/outputs/SNMOT-137
SNMOT-137_tracked.mp4 -> /content/drive/MyDrive/football/football/outputs/SNMOT-137
